In [1]:
from xmlrpc.server import SimpleXMLRPCServer
from xmlrpc.server import SimpleXMLRPCRequestHandler

import sqlite3
import threading


DB_PATH = "log.db"
_lock = threading.Lock()


class RequestHandler(SimpleXMLRPCRequestHandler):
    rpc_paths = ('/RPC2',)


server = SimpleXMLRPCServer(("localhost", 8018),
                            requestHandler=RequestHandler,
                            allow_none=True,
                            logRequests=False)

def _init_db():
    with sqlite3.connect(DB_PATH) as con:
        con.execute("""
            CREATE TABLE IF NOT EXISTS events (
                id         INTEGER PRIMARY KEY AUTOINCREMENT,
                ts_event   TEXT NOT NULL,   -- время события (ISO)
                event_type TEXT NOT NULL,   -- 'operation' | 'too_many_requests' | ...
                method     TEXT,            -- имя RPC-метода сервера
                duration   REAL,            -- длительность выполнения на сервере, сек
                status     TEXT,            -- 'ok' | 'error' | 'rate_limited'
                client     TEXT
            )
        """)
        con.execute("CREATE INDEX IF NOT EXISTS ix_events_type ON events(event_type)")
        con.execute("CREATE INDEX IF NOT EXISTS ix_events_ts   ON events(ts_event)")
        con.commit()

_init_db()


def log_event(ts_event, event_type, method=None, duration=None,
              status=None, client=None):
    with _lock, sqlite3.connect(DB_PATH) as con:
        con.execute(
            "INSERT INTO events(ts_event,event_type,method,duration,status,client)"
            " VALUES(?,?,?,?,?,?)",
            (ts_event, event_type, method, duration, status, client),
        )
        con.commit()
    return True


def get_events(method=None, event_type=None,
               ts_from=None, ts_to=None,
               min_duration=None, max_duration=None,
               limit=1000):
    sql = ("SELECT id, ts_event, event_type, method, duration, status, client "
           "FROM events WHERE 1=1")
    args = []
    if method       is not None: sql += " AND method = ?";     args.append(method)
    if event_type   is not None: sql += " AND event_type = ?"; args.append(event_type)
    if ts_from      is not None: sql += " AND ts_event >= ?";  args.append(ts_from)
    if ts_to        is not None: sql += " AND ts_event <= ?";  args.append(ts_to)
    if min_duration is not None: sql += " AND duration >= ?";  args.append(min_duration)
    if max_duration is not None: sql += " AND duration <= ?";  args.append(max_duration)
    sql += " ORDER BY id DESC LIMIT ?"; args.append(int(limit))

    with sqlite3.connect(DB_PATH) as con:
        rows = con.execute(sql, args).fetchall()
    keys = ("id","ts_event","event_type","method",
            "duration","status","client")
    return [dict(zip(keys, r)) for r in rows]


def get_events_df(method=None, event_type=None,
                  ts_from=None, ts_to=None,
                  min_duration=None, max_duration=None,
                  limit=1000):
    rows = get_events(method, event_type, ts_from, ts_to,
                      min_duration, max_duration, limit)
    if not rows:
        return {}
    cols = list(rows[0].keys())
    return {c: [r[c] for r in rows] for c in cols}


def clear_events():
    with _lock, sqlite3.connect(DB_PATH) as con:
        con.execute("DELETE FROM events")
        con.commit()
    return True

server.register_introspection_functions()


server.register_function(log_event, 'log_event')
server.register_function(get_events, 'get_events')
server.register_function(get_events_df, 'get_events_df')
server.register_function(clear_events, 'clear_events')

print(server.system_listMethods())

print("Listening on port 8018...")
server.serve_forever()

['clear_events', 'get_events', 'get_events_df', 'log_event', 'system.listMethods', 'system.methodHelp', 'system.methodSignature']
Listening on port 8018...


KeyboardInterrupt: 